In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
import random

PROJECT_DIR = Path.cwd().parent

MALAYSIA_DIR = (
    PROJECT_DIR
    / "datasets"
    / "dataset-malaysia"
    / "MunirahRosbi-Outdoor-Tenera-Oil-Palm-Fruit-Image"
)

TRAIN_DIR = MALAYSIA_DIR / "FFBtrain"
TEST_DIR = MALAYSIA_DIR / "FFBtest"

print("Train folder:", TRAIN_DIR.exists())
print("Test folder:", TEST_DIR.exists())

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent

DATASET_ROOT = PROJECT_DIR / "datasets" / "dataset-malaysia"

print("Dataset root ditemukan:", DATASET_ROOT.exists())
print("\nIsi folder dataset-malaysia:")

for item in DATASET_ROOT.iterdir():
    print(item.name, "| Folder:", item.is_dir())

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent

MALAYSIA_DIR = (
    PROJECT_DIR
    / "datasets"
    / "dataset-malaysia"
    / "MunirahRosbi-Outdoor-Tenera-Oil-Palm-Fruit-Image-b27ea70"
)

TRAIN_DIR = MALAYSIA_DIR / "FFBtrain"
TEST_DIR = MALAYSIA_DIR / "FFBtest"

print("Malaysia folder:", MALAYSIA_DIR.exists())
print("Train folder:", TRAIN_DIR.exists())
print("Test folder:", TEST_DIR.exists())

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random

classes_malaysia = [
    "TrainDamaged",
    "TrainEmpty",
    "TrainOverripe",
    "TrainRipe",
    "TrainUnripe"
]

fig, axes = plt.subplots(1, 5, figsize=(20, 5))

for ax, class_name in zip(axes, classes_malaysia):

    image_files = [
        p for p in (TRAIN_DIR / class_name).iterdir()
        if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ]

    image_path = random.choice(image_files)

    with Image.open(image_path) as image:
        ax.imshow(image.copy())

    ax.set_title(class_name.replace("Train", ""))
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from PIL import Image
import pandas as pd
from tqdm import tqdm

dimension_results_3 = []

for split_name, split_dir in [
    ("train", TRAIN_DIR),
    ("test", TEST_DIR)
]:
    for class_dir in split_dir.iterdir():

        if not class_dir.is_dir():
            continue

        for image_path in tqdm(
            list(class_dir.iterdir()),
            desc=f"Memeriksa {class_dir.name}"
        ):
            if image_path.suffix.lower() not in [
                ".jpg", ".jpeg", ".png"
            ]:
                continue

            try:
                with Image.open(image_path) as img:
                    width, height = img.size

                    dimension_results_3.append({
                        "Split": split_name,
                        "Kelas": class_dir.name,
                        "Nama File": image_path.name,
                        "Width": width,
                        "Height": height
                    })

            except Exception as e:
                print("Gagal membaca:", image_path.name, e)

df_dimensions_3 = pd.DataFrame(dimension_results_3)

print("Total gambar:", len(df_dimensions_3))

display(
    df_dimensions_3.groupby(
        ["Width", "Height"]
    ).size().reset_index(name="Jumlah Gambar")
)

In [ ]:
import cv2
import numpy as np
from tqdm import tqdm

quality_results_3 = []

for split_name, split_dir in [
    ("train", TRAIN_DIR),
    ("test", TEST_DIR)
]:
    for class_dir in split_dir.iterdir():

        if not class_dir.is_dir():
            continue

        for image_path in tqdm(
            list(class_dir.iterdir()),
            desc=f"Mengukur {class_dir.name}"
        ):
            if image_path.suffix.lower() not in [
                ".jpg", ".jpeg", ".png"
            ]:
                continue

            image = cv2.imread(str(image_path))

            if image is None:
                continue

            gray = cv2.cvtColor(
                image,
                cv2.COLOR_BGR2GRAY
            )

            brightness = np.mean(gray)
            contrast = np.std(gray)
            sharpness = cv2.Laplacian(
                gray,
                cv2.CV_64F
            ).var()

            quality_results_3.append({
                "Split": split_name,
                "Kelas": class_dir.name,
                "Nama File": image_path.name,
                "Brightness": brightness,
                "Contrast": contrast,
                "Sharpness": sharpness
            })

df_quality_3 = pd.DataFrame(quality_results_3)

print("Total gambar dianalisis:", len(df_quality_3))

display(
    df_quality_3.groupby("Split")[
        ["Brightness", "Contrast", "Sharpness"]
    ].mean().round(2)
)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

metrics_3 = [
    "Brightness",
    "Contrast",
    "Sharpness"
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, metric in zip(axes, metrics_3):
    sns.boxplot(
        data=df_quality_3,
        x="Split",
        y=metric,
        ax=ax
    )

    ax.set_title(f"Distribusi {metric}")
    ax.set_xlabel("Split")
    ax.set_ylabel(metric)

plt.suptitle(
    "Distribusi Kualitas Gambar Dataset Malaysia",
    fontsize=14
)

plt.tight_layout()
plt.show()

In [8]:
# Simpan hasil EDA Dataset Malaysia

RESULT_DIR_3 = PROJECT_DIR / "results" / "eda_dataset_malaysia"
RESULT_DIR_3.mkdir(parents=True, exist_ok=True)

# 1. Distribusi kelas
class_counts_3 = {
    "Damaged": [12, 3],
    "Empty": [9, 3],
    "Overripe": [60, 14],
    "Ripe": [160, 41],
    "Unripe": [130, 34]
}

df_class_distribution_3 = pd.DataFrame(
    [
        {
            "Kelas": kelas,
            "Train": jumlah[0],
            "Test": jumlah[1],
            "Total": sum(jumlah)
        }
        for kelas, jumlah in class_counts_3.items()
    ]
)

df_class_distribution_3.to_csv(
    RESULT_DIR_3 / "distribusi_kelas.csv",
    index=False
)

# 2. Dimensi gambar
df_dimensions_3.to_csv(
    RESULT_DIR_3 / "dimensi_gambar.csv",
    index=False
)

# 3. Kualitas gambar
df_quality_3.to_csv(
    RESULT_DIR_3 / "kualitas_gambar.csv",
    index=False
)

print("Seluruh hasil EDA Dataset 3 berhasil disimpan.")
print("Lokasi:", RESULT_DIR_3)

Seluruh hasil EDA Dataset 3 berhasil disimpan.
Lokasi: c:\Users\gabes\OneDrive\GABE_JUNTAK\SEMESTER 5\CERTAN\PRAKTIKUM\benchmark-tbs-rf-detr-vs-yolo\results\eda_dataset_malaysia
